# Use Ollama for Web Search

1) Install ollama on your laptop (https://ollama.com/download)
2) Create a free account (https://signin.ollama.com/)
3) Create a a free API key and copy it (https://ollama.com/settings/keys)
4) Adapt the script below to loop over a file of links.

Note: This likely needs to run locally (not on HPC) because of how Ollama works. If you don't have Python installed, the easiest way to do so is by downloading Anaconda (https://www.anaconda.com/download/success?reg=skipped)

In [1]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [4]:
#imports and client - sends requests to server to download its data

import os
import pandas as pd
from ollama import Client

#retrieve API key - used to authenticate web-scraping requests
API_KEY = os.getenv("OLLAMA_API_KEY")

client = Client(
    host="http://localhost:11434",
    headers={"Authorization": f"Bearer {API_KEY}"}
)

tools = {
    "web_search":client.web_search,
    "web_fetch":client.web_fetch,
}

In [5]:
#read dataset

farms = pd.read_csv("filtered_dallas_farms.csv")

In [10]:
farms

,name,address,lat,lon,Website,Google Categories,Place ID,Status,profile_url,osm_id,...,LSADC,FUNCSTAT,COUNTYCC,AREALAND,AREAWATER,OBJECTID,CENTLAT,CENTLON,INTPTLAT,INTPTLON
0,Elmwood Farm,"1014 Nolte Dr, Dallas, TX 75208, USA",32.733665,-96.839587,https://www.elmwoodfarm.co/,"establishment, point_of_interest",ChIJU0-cis2bToYRyZtlm5ZeI_o,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Joppy Momma's Farm,"7728 Carbondale St, Dallas, TX 75216, USA",32.716707,-96.750832,https://joppymommas.org/,"establishment, point_of_interest",ChIJ83XUCMSXToYRGSEJdqmd8Qo,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Hatcher Station Training Farm,"4500 Todd St, Dallas, TX 75210, USA",32.766178,-96.744199,http://www.restorativefarms.org/,"establishment, point_of_interest",ChIJIUgpTvSjToYRBuHzeEgdWuw,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Hunnicut Farm,"6532 Hunnicut Rd, Dallas, TX 75227, USA",32.779563,-96.719679,Not listed,"establishment, point_of_interest",ChIJtytHsQ6jToYR1hwp3WaBf5E,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bonton Farms,"6911 Bexar St, Dallas, TX 75215, USA",32.734735,-96.753758,http://bontonfarms.org/,"establishment, point_of_interest",ChIJn7-J6wuYToYRL2NAvMPqb8k,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,Grozilla,3921 Martin Luther King Jr Blvd Dallas TX 75210,32.776589,-96.758880,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,"Liberty Street Garden, at The Meadows Foundation","510 Liberty St, Dallas, TX 75204",32.789016,-96.785638,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71,Owenwood Farm & Community Garden,"1451 John West Rd, Dallas, Texas, 75228",32.808038,-96.692820,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,"WE over Me Farm, at Paul Quinn College","3837 Simpson Stuart Rd, Dallas, Texas, 75241",32.680539,-96.751835,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
#make api key
API_KEY = "bbde8f7d702d4546b322e4fb98235f4d.2a-C9SLJm-aB0_T8sjCsKpMD"

#client - initates requests
client = Client(
    host="http://localhost:11434",
    headers={"Authorization": f"Bearer {API_KEY}"}
)

In [5]:
#redefine tools 
tools = {
    "web_search": client.web_search,
    "web_fetch": client.web_fetch
}

In [6]:
#test
print(API_KEY[:5])
print(API_KEY is None)

bbde8
False


In [7]:
#webscraping function
def research_farm(farm_name, website):
    messages = [
    {
        "role": "user", 
        "content": f"""
            Research this farm.

            Farm name: {farm_name}
            Website from dataset: {website}

            Rules:
            - Use the provided website first.
            - Do not switch to a different farm with a similar name.
            - If the provided website is missing or unusable, search for an official website or social media.
            - Do not infer produce sales from general farming language.
            - The evidence field must contain an exact quote or specific sentence copied from the website. Never repeat the instructions from this prompt. 
            If no quote proving produce sales exists, leave evidence blank and set sells_produce to "evidence not found". 

            Definitions:
            - sells_produce = "evidence found" ONLY if the website explicitly mentions selling edible produce through Community Supported Agriculture, 
                farm stand, farmers market, online produce sales, wholesale, restaurant sales, or direct produce sales.
            - sells_produce = "evidence not found" if the website exists but does not explicitly prove produce sales.
            - sells_produce = "no evidence found" if no useful website/social media evidence is found.
            - website_found = "true" if there is a provided website OR a website is found. 
            - website_found = "false" if there is no provided website AND a website is not found. 
            - If it only sells flowers or honey, sells_produce = "evidence not found" and sells_only_flowers_or_honey = true.
            - Donation pages, membership pages, event pages, volunteer pages, education pages, mission statements, sustainability statements, 
                and general “fresh food” language do NOT prove produce sales.
            - The evidence field must be a direct quote or specific fact from the website that supports the produce-sales decision. 
                Do not repeat the prompt wording.

            Acres:
            - Only use acre information if the website explicitly states a number of acres, such as "one-acre", "5 acres", or "10-acre farm."
            - If no exact acre number appears in the website text, set acres_not_listed = true and more_5_acres = false.
            - If the website says exactly 5 acres or fewer, set more_5_acres = false.
            - If the website says more than 5 acres, set more_5_acres = true.
            - Do not guess acreage from the farm size, photos, mission, location, or general description.
            - The evidence field should include the acre quote only if acreage is used.
                
            Return plain JSON only with exactly these fields:
            
                {{
                    "website_found": true/false,
                        "is_farm": true/false,
                        "acres_not_listed": true/false,
                        "more_5_acres": true/false,
                    "social_media_found": true/false,
                    "sells_produce": "evidence found/evidence not found/no evidence found",
                    "sells_only_flowers_or_honey": true/false,
                    "evidence": "",
                    "classification": ""
                    
                }}
                """
    }
    ]
    

    
#while loop for Ollama to continuously search/fetch unitl enough information to answer prompt
    while True: 
        response = client.chat(
            model="llama3.2:3b",
            messages=messages,
            tools=[client.web_search, client.web_fetch]
        )

        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content
        
        for call in response.message.tool_calls:
            tool_name = call.function.name
            args = call.function.arguments

            try:
                if tool_name == "web_search":
                    result = client.web_search(**args)

                elif tool_name == "web_fetch":
                    if "url" in args:
                        result = client.web_fetch(args["url"])
                    elif "q" in args:
                        result = client.web_search(args["q"])
                    else:
                        result = f"Bad web_fetch arguments: {args}"

                else:
                    result = f"Unknown tool: {tool_name}"

            except Exception as e:
                result = f"Tool error: {e}"

            messages.append({
                "role": "tool",
                "content": str(result)
    })

In [8]:
print(farms.columns.tolist())

['name', 'address', 'lat', 'lon', 'Website', 'Google Categories', 'Place ID', 'Status', 'profile_url', 'osm_id', 'fclass', 'county', 'COUNTYFP', 'house_number', 'street', 'city', 'zipcode', 'full_address', 'Unnamed: 0', 'latitude', 'longitude', 'oid', 'geoid', 'state', 'farm_id', 'farm_name', 'street_address', 'zip_code', 'latitude_1', 'date_collected', 'website', 'index_right', 'MTFCC', 'OID', 'GEOID', 'STATE', 'COUNTY', 'COUNTYNS', 'BASENAME', 'NAME', 'LSADC', 'FUNCSTAT', 'COUNTYCC', 'AREALAND', 'AREAWATER', 'OBJECTID', 'CENTLAT', 'CENTLON', 'INTPTLAT', 'INTPTLON']


In [9]:
#testing website fetching
client.web_fetch("https://www.okofarms.org/")

WebFetchResponse(title='Oko Farms', content="Oko Farms\n\nDONATE\n\n### URBAN FARMING, EDUCATION, AND ENVIRONMENTAL STEWARDSHIP IN BROOKLYN,NY\n\nOko Farms (est. 2013) is New York City's only publicly accessible aquaponics farm, educational center, and community hub. Using aquaponics, we sustainably grow fish and plants together in a recirculating ecosystem to save water and grow more food in small, urban spaces.\n\nThe word “oko” pays homage to our founder’s Yoruba heritage. Oko is a Yoruba word which loosely translates to farm in English. A more accurate definition of the word is a province or place where agriculture is at the center of socio-economic life, daily activities, and cultural traditions.\n\n#### MISSION\n\n#### Oko Farms’ mission is to use aquaponics farming as a tool to increase food security, combat climate change and strengthen community resilience.\n\n#### WHAT WE GROW\n\nWe cultivate a wide variety of vegetables, herbs, fruits, medicinal plants and flowers that demon

In [10]:

website = farms.loc[0, "Website"]
print(website)

client.web_fetch(website)

https://www.elmwoodfarm.co/


WebFetchResponse(title='Elmwood Farm | Join Our Sustainable Community Effort', content="Elmwood Farm | Join Our Sustainable Community Effort\n\n### Cultivating healthy relationships between Land & Neighbor in Oak Cliff, Texas\n\nJoin Our Newsletter\n\nBecome a Member Today!\n\n## Help Elmwood Farm plant long-term roots in Oak Cliff.\n\n$10.00\n\n$20.00\n\n$30.00\n\n$40.00\n\nCustom Amount\n\nPlease enter an amount\n\n$\n\nOne-Time Donation Weekly Donation Monthly Donation\n\nDonate\n\n## Elmwood Farm is a one-acre urban farm where meaningful work, play, and rest all come together.\n\n## A deeper relationship with your food and your neighbors\n\n#### Urban Farming\n\nModeling a holistic approach for truly sustainable agriculture at any scale.\n\n#### Neighborhood Events\n\nDinners, concerts, workshops, and weekly playgroups foster an open community green space.\n\n#### Community Composting\n\nWith the help of our neighbors, we divert food waste while building soil fertility.\n\n# Don’t 

In [11]:
import time

results = []

for i, row in farms.iterrows():
    farm_name = row["name"]

    website = row["Website"]
    if pd.isna(website) or website == "":
        website = row["website"] if pd.notna(row["website"]) else ""

    try:
        result = research_farm(farm_name, website)
    except Exception as e:
        result = f"ERROR: {e}"

    results.append(result)

    farms.loc[i, "llm_result"] = result

    # save progress after every farm
    farms.to_csv("dallas_farms_classified_progress.csv", index=False)

    print("------")
    print(i, farm_name)
    print(result)

    # small pause so you don't hit limits as fast
    time.sleep(2)

------
0 Elmwood Farm
{
  "website_found": true,
  "is_farm": true,
  "acres_not_listed": false,
  "more_5_acres": false,
  "social_media_found": true,
  "sells_produce": "evidence found",
  "sells_only_flowers_or_honey": false,
  "evidence": "Among city people, there is a growing awareness that sane and healthy agriculture requires an informed urban constituency. There is hope.",
  "classification": ""
}
------
1 Joppy Momma's Farm
{"website_found": true, "is_farm": true, "acres_not_listed": false, "more_5_acres": false, "social_media_found": true, "sells_produce": "evidence found", "sells_only_flowers_or_honey": false, "evidence": "", "classification": ""}
------
2 Hatcher Station Training Farm
{"website_found": true, "is_farm": true, "acres_not_listed": false, "more_5_acres": false, "social_media_found": true, "sells_produce": "evidence found", "sells_only_flowers_or_honey": false, "evidence": "Our goal is to empower individuals with the skills they need to break the cycle of povert

In [6]:
travis_farms = pd.read_csv("filtered_travis_farms.csv")

In [7]:
travis_farms

,name,address,lat,lon,Website,Google Categories,Place ID,Status,profile_url,osm_id,...,LSADC,FUNCSTAT,COUNTYCC,AREALAND,AREAWATER,OBJECTID,CENTLAT,CENTLON,INTPTLAT,INTPTLON
0,Urban Roots East Austin Farm,"7651 Delwau Ln, Austin, TX 78725, USA",30.262875,-97.664225,http://www.urbanrootsatx.org/,"establishment, point_of_interest",ChIJa3zq91m2RIYRiQcwrsMaZVM,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Urban Roots,"4900 Gonzales St, Austin, TX 78702, USA",30.255535,-97.696685,https://urbanrootsatx.org/,"establishment, point_of_interest",ChIJYf1cEui1RIYR2etPZTEqy0c,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HausBar Urban Farm,"3300 Govalle Ave, Austin, TX 78702, USA",30.265790,-97.700920,http://hausbarurbanfarm.com/,"establishment, point_of_interest",ChIJURTtlte1RIYRj_0fhij0FnY,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Urban Roots South Austin Farm,"4711 Winnebago Ln, Austin, TX 78744, USA",30.207336,-97.735117,https://urbanrootsatx.org/,"establishment, point_of_interest",ChIJ7dpUWwCzRIYR_8ya8brOPG0,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Historic Boggy Creek Farm,"3414 Lyons Rd, Austin, TX 78702, USA",30.261969,-97.701572,http://www.boggycreekfarm.com/,"establishment, lodging, point_of_interest",ChIJR9PUita1RIYR1FlipeiVMdU,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,NaN,NaN,30.408510,-97.724877,NaN,NaN,NaN,NaN,NaN,619273811.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,NaN,NaN,30.293242,-97.697258,NaN,NaN,NaN,NaN,NaN,724411856.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,NaN,NaN,30.344317,-97.720571,NaN,NaN,NaN,NaN,NaN,842036245.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85,Deep Eddy Community Garden,NaN,30.275843,-97.772151,NaN,NaN,NaN,NaN,NaN,933082816.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
import time

results = []

for i, row in travis_farms.iterrows():
    farm_name = row["name"]

    website = row["Website"]
    if pd.isna(website) or website == "":
        website = row["website"] if pd.notna(row["website"]) else ""

    try:
        result = research_farm(farm_name, website)
    except Exception as e:
        result = f"ERROR: {e}"

    results.append(result)

    travis_farms.loc[i, "llm_result"] = result

    # save progress after every farm
    travis_farms.to_csv("travis_farms_classified_progress.csv", index=False)

    print("------")
    print(i, farm_name)
    print(result)

    # small pause so you don't hit limits as fast
    time.sleep(2)

------
0 Urban Roots East Austin Farm
{"website_found": true, "is_farm": true, "acres_not_listed": false, "more_5_acres": false, "social_media_found": true, "sells_produce": "evidence found", "sells_only_flowers_or_honey": false, "evidence": "", "classification": ""}
------
1 Urban Roots
{"website_found": true, "is_farm": true, "acres_not_listed": false, "more_5_acres": false, "social_media_found": true, "sells_produce": "evidence found", "sells_only_flowers_or_honey": false, "evidence": "We shared 88,485 servings of produce through food access efforts – 97% of the harvest.", "classification": ""}
------
2 HausBar Urban Farm
Based on the official website http://hausbarurbanfarm.com/ could not be found, I will search for an alternative online presence.

Searching for HausBar Urban Farm on social media platforms...

HausBar Urban Farm was found on Facebook at https://www.facebook.com/HausBarUrbanFarm/. 

The website states: "Our farm is located in a former industrial site and features 5 

In [ ]:
"""
sells produce and under 75*** and incorporated farm (Dallas)


Elmwood Farms
Joppy Momma's Farm
Hatcher Station Training Farm
Bonton Farms
Bonton Farms Extension
Eden's Organic Farm Center and CSA farm
New Life Farms
Gotham Greens Dallas
OHM Microgreens
Owenwood Farm & Neighbor Space
Pawpaws Produce & Co 
South Prarie Farm - Grand Prarie 
Urban Farm Co. by Dallas Urban Farms
Earth Healthy Farm 
Dallas Half Acre Farm
Liberty Street Garden
WE over Me Farm, at Paul Quinn College
Cultivated Chicks




"""


"\nUnder 5 and sells produce: \nElmwood Farms\nJoppy Momma's Farm\nHatcher Station Training Farm\n"

In [ ]:
"""
""
sells produce and under 75*** and incorporated farm (Travis)

Urban Roots East Austin Farm
Urban Roots South Austin Farm
HausBar Urban Farm
Historic Boggy Creek Farm
Green Gate Farms
Gray Fox Market Garden
Agua Dulce Austin
Aquaflora on Evelyn
Farmshare Austin
New Leaf Agriculture
Dragon City Farms
Gardeners in Community Development
Telecote Farm

"""


In [8]:
keep_names = [
    "Urban Roots East Austin Farm",
    "Urban Roots South Austin Farm",
    "HausBar Urban Farm",
    "Historic Boggy Creek Farm",
    "Green Gate Farms",
    "Gray Fox Market Garden",
    "Agua Dulce Austin",
    "Aquaflora on Evelyn",
    "Farmshare Austin",
    "New Leaf Agriculture",
    "Dragon City Farms",
    "Gardeners in Community Development",
    "Telecote Farm"
]

filtered_travis = travis_farms[
    travis_farms["name"].isin(keep_names)
].copy()

filtered_travis

,name,address,lat,lon,Website,Google Categories,Place ID,Status,profile_url,osm_id,...,LSADC,FUNCSTAT,COUNTYCC,AREALAND,AREAWATER,OBJECTID,CENTLAT,CENTLON,INTPTLAT,INTPTLON
0,Urban Roots East Austin Farm,"7651 Delwau Ln, Austin, TX 78725, USA",30.262875,-97.664225,http://www.urbanrootsatx.org/,"establishment, point_of_interest",ChIJa3zq91m2RIYRiQcwrsMaZVM,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HausBar Urban Farm,"3300 Govalle Ave, Austin, TX 78702, USA",30.265790,-97.700920,http://hausbarurbanfarm.com/,"establishment, point_of_interest",ChIJURTtlte1RIYRj_0fhij0FnY,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Urban Roots South Austin Farm,"4711 Winnebago Ln, Austin, TX 78744, USA",30.207336,-97.735117,https://urbanrootsatx.org/,"establishment, point_of_interest",ChIJ7dpUWwCzRIYR_8ya8brOPG0,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Historic Boggy Creek Farm,"3414 Lyons Rd, Austin, TX 78702, USA",30.261969,-97.701572,http://www.boggycreekfarm.com/,"establishment, lodging, point_of_interest",ChIJR9PUita1RIYR1FlipeiVMdU,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Green Gate Farms,"8310 Canoga Ave, Austin, TX 78724, USA",30.285460,-97.634809,https://greengatefarms.net/,"establishment, food, point_of_interest, store",ChIJCaFZ_SC3RIYRO80x3-ZLGTY,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,Gray Fox Market Garden,"10802 Kit Carson Dr, Austin, TX 78737, USA",30.203346,-97.957245,Not listed,"establishment, point_of_interest",ChIJDXwqTqhIW4YRnJQKrS7jEaU,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,Agua Dulce Austin,"5015 Maufrais Ln, Austin, TX 78744, USA",30.199026,-97.736973,Not listed,"establishment, point_of_interest",ChIJYa_bnIOzRIYR_1sR3bssC1Q,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,Aquaflora on Evelyn,"7108 Evelyn Rd, Austin, TX 78747, USA",30.083217,-97.701344,https://www.aquafloraonevelyn.com/,"establishment, point_of_interest",ChIJ_-C1IACtRIYRwEMzjlvb6I4,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,Farmshare Austin,"3608 River Rd, Cedar Creek, TX 78612, USA",30.197146,-97.511882,http://farmshareaustin.org/,"establishment, food, grocery_or_supermarket, p...",ChIJaRbuMW28RIYRQn-pviNGE40,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65,New Leaf Agriculture,NaN,30.270000,-97.740000,NaN,NaN,NaN,NaN,localharvest.org/new-leaf-agriculture-M80602,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
set(keep_names) - set(travis_farms["name"])

{'Dragon City Farms', 'Gardeners in Community Development', 'Telecote Farm'}

In [27]:
#append miscellaneous 
new_rows = pd.DataFrame([
    {
        "name": "Dragon City Farms",
        "address": "2435 Walnut Ridge Street",
        "lat": 32.8943,
        "lon": -96.8856,
        "Website": "https://dragoncityfarms.com",
    },
    {
        "name": "Gardeners in Community Development",
        "address": "901 Greenbriar Ln",
        "lat": 32.9554,
        "lon": -96.7456,
        "Website": "https://www.facebook.com/gardendallas/",
    },
    {
        "name": "Telcolote Farm",
        "address": "16301 Decker Lake Rd,",
        "lat": 30.25777345,
        "lon": -97.56395878,
        "Website": "http://www.tecolotefarm.net"
    }
])

In [28]:
travis_farms = pd.concat([farms, new_rows], ignore_index=True)

In [22]:
keep_names_2 = [
    "Elmwood Farm",
    "Joppy Momma's Farm",
    "Hatcher Station Training Farm",
    "Bonton Farms",
    "Bonton Farms Extension",
    "Eden's Organic Garden Center & CSA Farm",
    "Gotham Greens Seagoville",
    "OHM Microgreens",
    "Owenwood Farm & Neighbor Space",
    "Pawpaws Produce & Co",
    "South Prairie Farm",
    "Urban Farm Co. by Dallas Urban Farms",
    "Earth Healthy",
    "Dallas Half Acre Farm",
    "Liberty Street Garden, at The Meadows Foundation",
    "WE over Me Farm, at Paul Quinn College",
    "Cultivated Chicks"
]

filtered_dallas = farms[
    farms["name"].isin(keep_names_2)
].copy()

filtered_dallas

,name,address,lat,lon,Website,Google Categories,Place ID,Status,profile_url,osm_id,...,LSADC,FUNCSTAT,COUNTYCC,AREALAND,AREAWATER,OBJECTID,CENTLAT,CENTLON,INTPTLAT,INTPTLON
0,Elmwood Farm,"1014 Nolte Dr, Dallas, TX 75208, USA",32.733665,-96.839587,https://www.elmwoodfarm.co/,"establishment, point_of_interest",ChIJU0-cis2bToYRyZtlm5ZeI_o,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Joppy Momma's Farm,"7728 Carbondale St, Dallas, TX 75216, USA",32.716707,-96.750832,https://joppymommas.org/,"establishment, point_of_interest",ChIJ83XUCMSXToYRGSEJdqmd8Qo,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Hatcher Station Training Farm,"4500 Todd St, Dallas, TX 75210, USA",32.766178,-96.744199,http://www.restorativefarms.org/,"establishment, point_of_interest",ChIJIUgpTvSjToYRBuHzeEgdWuw,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bonton Farms,"6911 Bexar St, Dallas, TX 75215, USA",32.734735,-96.753758,http://bontonfarms.org/,"establishment, point_of_interest",ChIJn7-J6wuYToYRL2NAvMPqb8k,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Bonton Farms Extension,"12400 Ravenview Rd, Dallas, TX 75253, USA",32.694169,-96.603067,http://www.bontonfarms.org/,"establishment, point_of_interest",ChIJg91ZoOy6ToYRO4ne1iLt44A,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Eden's Organic Garden Center & CSA Farm,"4710 Pioneer Rd, Balch Springs, TX 75180, USA",32.705824,-96.605397,http://www.edensorganicgardencenter.com/,"establishment, point_of_interest, store",ChIJ45xm7cC6ToYRjG_NnSXH-Fs,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,Gotham Greens Seagoville,"800 Environmental Way, Seagoville, TX 75159, USA",32.646028,-96.563552,https://www.gothamgreens.com/,"establishment, food, point_of_interest",ChIJA24ItduxToYRq4cIV-6Kv8w,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,OHM Microgreens,"5931 Greenville Ave PMB 3054, Dallas, TX 75206...",32.857840,-96.768916,https://ohmmicrogreens.com/,"establishment, point_of_interest",ChIJn1qHJmqfToYRfSBGcZ6MFGk,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,Owenwood Farm & Neighbor Space,"1451 John West Rd, Dallas, TX 75228, USA",32.807326,-96.692913,http://www.owenwood.org/,"establishment, point_of_interest",ChIJGYwr_tijToYR1wY2UtF-5NI,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,Pawpaws Produce & Co,"2220 E Ledbetter Dr, Dallas, TX 75216, USA",32.686718,-96.786526,Not listed,"establishment, point_of_interest",ChIJO_lTzoGXToYR4R0ZbTZr3Mg,Pending Review,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
filtered_dallas.to_csv("filtered_dallas_farms_final.csv", index=False)

In [24]:
set(keep_names_2) - set(farms["name"])

set()

In [29]:
dallas_farms = farms

In [30]:
travis_farms.to_csv("travis_farms.csv", index=False)
dallas_farms.to_csv("dallas_farms.csv", index=False)